# Aerial Object Detection using HerdNet
https://github.com/Alexandre-Delplanque/HerdNet

look at the phd thesis of Alexandre Delplanque for more details: https://orbi.uliege.be/handle/2268/299905
The phd thesis is at: https://orbi.uliege.be/bitstream/2268/321482/1/20240826_Thesis_Alexandre_Delplanque.pdf

The Master Thesis shows the application of HerdNet on Marine Iguanas:
https://doi.org/10.6084/m9.figshare.30719999


The self contained installation within this notbook is a bit tricky. Have a look at the colab version of a similar notebook for a more straightforward installation: [Modified Notebook from Original Author](https://github.com/cwinkelmann/HerdNet/blob/main/notebooks/demo-training-testing-herdnet_local.ipynb)

## Setup

This notebook requires the **fit-training** conda environment which
includes `animaloc` (HerdNet). See `INSTALLATION_INSTRUCTIONS.md`.

```bash
conda activate fit-training
```

In [1]:
# Colab only — install dependencies if not already available
# import sys

# !git clone  https://github.com/cwinkelmann/usde-innovations-applications-forest-it.git fit-course
# !cd fit-course && git pull
# !pip install -e "./fit-course[herdnet,dev]"
#
# # sys.path is handled by pip install -e above — no manual append needed

In [ ]:
import os
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# num_workers > 0 speeds up data loading; Windows Jupyter can't fork workers safely
num_workers = 0 if os.name == "nt" else 2

print(f"Using device: {device}  |  num_workers: {num_workers}")

In [3]:
# configure the logger
from loguru import logger

logger.disable("animaloc")


In [4]:
# Verify animaloc is installed (comes with fit-training env)
import animaloc
print(f"animaloc {animaloc.__version__}")

animaloc 0.2.1


Load a seperate Sample of the test set

In [5]:
from huggingface_hub import hf_hub_download, snapshot_download
from pathlib import Path

local_dir = Path("./models").resolve()

# Download model weights and config
model_path = hf_hub_download(
    repo_id="karisu/HerdNet",
    filename="general_2022/20220413_HerdNet_General_dataset_2022.pth",
    local_dir=local_dir
)
config_path = hf_hub_download(
    repo_id="karisu/HerdNet",
    filename="general_2022/config.yaml",
    local_dir=local_dir
)
print(f"Model: {model_path}\nConfig: {config_path}")

Model: /Users/christian/work/hnee/usde-innovations-applications-forest-it/week1/practicals/models/general_2022/20220413_HerdNet_General_dataset_2022.pth
Config: /Users/christian/work/hnee/usde-innovations-applications-forest-it/week1/practicals/models/general_2022/config.yaml


Load a part of the General Dataset which is hosted on Huggingface
https://dataverse.uliege.be/dataset.xhtml?persistentId=doi:10.58119/ULG/MIRUU5


In [6]:

# Download a small sample of the test set
sample_data_path = snapshot_download(
    repo_id="karisu/General_Dataset",
    repo_type="dataset",
    local_dir="./data",
    allow_patterns=["test_sample/*"],
    revision="main",
)
test_sample_annotation_path = hf_hub_download(
    repo_id="karisu/General_Dataset",
    repo_type="dataset",
    filename="test_sample.csv",
    local_dir="./data",
)
print(f"Test data: {sample_data_path}")

Fetching ... files: 0it [00:00, ?it/s]

Test data: /Users/christian/work/hnee/usde-innovations-applications-forest-it/week1/practicals/data


In [7]:
# Training, validation and test datasets
import albumentations as A


from animaloc.datasets import CSVDataset
from animaloc.data.transforms import MultiTransformsWrapper, DownSample, PointsToMask, FIDT

patch_size = 512
num_classes = 7
down_ratio = 2

test_dataset = CSVDataset(
    csv_file = './data/test_sample.csv',
    root_dir = './data/test_sample',
    albu_transforms = [A.Normalize(p=1.0)],
    end_transforms = [DownSample(down_ratio=down_ratio, anno_type='point')]
    )

In [8]:
# Test installation
# Set the seed
from animaloc.utils.seed import set_seed
set_seed(9292)


In [ ]:
from torch.utils.data import DataLoader

test_dataloader = DataLoader(dataset=test_dataset, batch_size=1, shuffle=False, num_workers=num_workers)

The Loss Wrapper is needed to evaluate the model against a known test-set

In [ ]:
from animaloc.models import HerdNet, LossWrapper
from animaloc.train.losses import FocalLoss
from torch import Tensor
from torch.nn import CrossEntropyLoss

herdnet = HerdNet(num_classes=num_classes, down_ratio=down_ratio).to(device)

weight = Tensor([0.1, 1.0, 2.0, 1.0, 6.0, 12.0, 1.0]).to(device)

losses = [
    {'loss': FocalLoss(reduction='mean'), 'idx': 0, 'idy': 0, 'lambda': 1.0, 'name': 'focal_loss'},
    {'loss': CrossEntropyLoss(reduction='mean', weight=weight), 'idx': 1, 'idy': 1, 'lambda': 1.0, 'name': 'ce_loss'},
]

# LossWrapper is needed by the Evaluator even when not training
herdnet = LossWrapper(herdnet, losses=losses)

In [ ]:
from torch.optim import Adam
from animaloc.train import Trainer
from animaloc.eval import PointsMetrics, HerdNetStitcher, HerdNetEvaluator
from animaloc.utils.useful_funcs import mkdir

work_dir = './output'
mkdir(work_dir)

metrics = PointsMetrics(radius=20, num_classes=num_classes)

stitcher = HerdNetStitcher(
    model=herdnet,
    size=(patch_size, patch_size),
    overlap=160,
    down_ratio=down_ratio,
    reduction='mean',
    device_name=str(device),
)

## Running HerdNet

At first a pretrained model is run against the Test Subset on African Megafauna.
Then it shown how to train the model.


### Test the pretrained model

A downloaded model is run against a small data.

In [ ]:
from animaloc.models import load_model

pth_path = './models/general_2022/20220413_HerdNet_General_dataset_2022.pth'
herdnet = load_model(herdnet, pth_path=pth_path)
herdnet = LossWrapper(herdnet, losses=losses)

In [ ]:
# Create output folder
test_dir = './test_output'
mkdir(test_dir)

In [ ]:
# Create an Evaluator
test_evaluator = HerdNetEvaluator(
    model=herdnet,
    dataloader=test_dataloader,
    metrics=metrics,
    stitcher=stitcher,
    work_dir=test_dir,
    header='test',
    device_name=str(device),
    )

In [ ]:
# Start testing
test_f1_score = test_evaluator.evaluate(returns='f1_score')

In [ ]:
# Print global F1 score (%)
print(f"F1 score = {test_f1_score * 100:0.0f}%")


In [ ]:
# Get the detections
detections = test_evaluator.results
detections

## Running HerdNet on Eikelboom aerial imagery

The General model above was tested on its own dataset. How does it perform
on **Eikelboom aerial imagery** — oblique photos from manned aircraft over
Kenyan savanna (elephants, giraffes, zebras)?

This connects to Practical 6a (Domain Shift), where MegaDetector failed
on the same images. HerdNet was trained on aerial data, so it should do better.


In [ ]:
import pandas as pd
from pathlib import Path

EIKELBOOM_DIR = Path("./data/eikelboom_test")
EIKELBOOM_DIR.mkdir(parents=True, exist_ok=True)

ANNOTATION_CSV = Path("../data/eikelboom/annotations_test.csv")
IMG_DIR = Path("../data/eikelboom/test")

if ANNOTATION_CSV.exists():
    eik_gt = pd.read_csv(ANNOTATION_CSV, header=None,
                         names=["file", "x1", "y1", "x2", "y2", "species"])
    eik_gt["filename"] = eik_gt["file"].apply(lambda x: Path(x).name)
    eik_gt["x"] = (eik_gt["x1"] + eik_gt["x2"]) / 2
    eik_gt["y"] = (eik_gt["y1"] + eik_gt["y2"]) / 2
    print(f"Eikelboom test: {eik_gt['filename'].nunique()} images, {len(eik_gt)} annotations")
    print(f"Species: {eik_gt['species'].value_counts().to_dict()}")
else:
    print(f"Eikelboom data not found at {ANNOTATION_CSV}")
    print("Run the domain shift notebook first, or download from 4TU.ResearchData")

In [ ]:
# Prepare Eikelboom data in HerdNet CSV format
# HerdNet expects: images, x, y, labels

eik_herdnet_csv = EIKELBOOM_DIR / "eikelboom_herdnet.csv"

if ANNOTATION_CSV.exists():
    herdnet_rows = []
    for _, row in eik_gt.iterrows():
        herdnet_rows.append({
            "images": row["filename"],
            "x": int(row["x"]),
            "y": int(row["y"]),
            "labels": 6,  # Map all species to class 6 (Elephant in General model)
        })
    eik_herdnet_df = pd.DataFrame(herdnet_rows)
    eik_herdnet_df.to_csv(eik_herdnet_csv, index=False)
    print(f"HerdNet CSV: {len(eik_herdnet_df)} annotations -> {eik_herdnet_csv}")


In [ ]:
# Run HerdNet General model on Eikelboom test images
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra
from animaloc.utils.inference import inference

if not ANNOTATION_CSV.exists():
    print("Skipping: Eikelboom data not found. Run the domain-shift notebook first.")
else:
    GlobalHydra.instance().clear()

    config_dir = str(Path("models/general_2022/").resolve())

    with initialize_config_dir(config_dir=config_dir, version_base="1.1"):
        cfg_eik = compose(config_name="config", overrides=[
            f"model.load_from=models/general_2022/20220413_HerdNet_General_dataset_2022.pth",
            f"datasets.test.csv_file={eik_herdnet_csv}",
            f"datasets.test.root_dir={IMG_DIR}",
        ])

    eik_detections = inference(cfg_eik, plain_inference=False, vis_detections=True)
    print(f"Eikelboom detections: {len(eik_detections)}")

In [ ]:
# Visualize HerdNet detections on Eikelboom images
if ANNOTATION_CSV.exists() and len(eik_detections) > 0:
    top_images = eik_gt.groupby("filename").size().sort_values(ascending=False).head(4).index

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    for ax, fname in zip(axes.flat, top_images):
        img_path = IMG_DIR / fname
        if not img_path.exists():
            continue
        ax.imshow(Image.open(img_path))
        ax.set_title(fname, fontsize=9)

        img_gt = eik_gt[eik_gt["filename"] == fname]
        ax.scatter(img_gt["x"], img_gt["y"], c="lime", s=40,
                   marker="o", edgecolors="black", linewidths=1,
                   label=f"GT ({len(img_gt)})")

        img_det = eik_detections[eik_detections["images"] == fname]
        if len(img_det) > 0:
            ax.scatter(img_det["x"], img_det["y"], c="red", s=40,
                       marker="x", linewidths=2,
                       label=f"HerdNet ({len(img_det)})")

        ax.legend(loc="upper right", fontsize=8)
        ax.axis("off")

    plt.suptitle("HerdNet General model on Eikelboom aerial imagery", fontsize=14)
    plt.tight_layout()

HerdNet was trained on nadir aerial imagery of African megafauna, so it
should detect animals from above more effectively than MegaDetector.
However, the Eikelboom images are **oblique** (side-facing from aircraft),
which is a different geometric domain even within aerial imagery.

Compare these results with the AerialDetector + SAHI results from
Practical 6a to see how bounding-box detection vs. point-based detection
performs on the same images.


## Inferencing using a config file
Here just the model and configuration is loaded making the process transferable. Any config can be overridden.

In [ ]:
# Clear any previous Hydra state (important when re-running cells in a notebook)
GlobalHydra.instance().clear()

config_dir = str(Path.cwd() / "models/general_2022")

with initialize_config_dir(config_dir=config_dir, version_base="1.1"):
    cfg = compose(config_name="config", overrides=[
        "model.load_from=./models/general_2022/20220413_HerdNet_General_dataset_2022.pth",
        "datasets.test.root_dir=./data/test_sample/",
    ])

# Inspect the full config used for training
print(OmegaConf.to_yaml(cfg))

In [ ]:
# Run inference
detections = inference(cfg, plain_inference=True, vis_detections=True)

In [ ]:
# Show results
print(f"Total detections: {len(detections)}")
detections.head(10)

### Visualise detections

In [ ]:
%matplotlib inline

# Load data
gt_df = pd.read_csv("data/test_sample.csv")
det_df = pd.read_csv("detections.csv")

# First image
image_stem = "02033bf9b6c41f5815072434f8d61707cc8ea1fb"
image_path = f"data/test_sample/{image_stem}.jpeg"

gt = gt_df[gt_df["images"].str.startswith(image_stem)]
det = det_df[det_df["images"] == f"{image_stem}.jpeg"].dropna(subset=["x", "y"])

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(Image.open(image_path))
ax.scatter(gt["x"], gt["y"], c="lime", s=100, marker="o",
           edgecolors="black", linewidths=1.5, label=f"Ground Truth ({len(gt)})")
ax.scatter(det["x"], det["y"], c="red", s=100, marker="x",
           linewidths=2, label=f"Detections ({len(det)})")
ax.set_title(f"{image_stem}\nGT: {len(gt)} | Det: {len(det)}")
ax.legend(loc="upper right")
ax.axis("off")
plt.tight_layout()
plt.savefig("detection_plot.png", dpi=150, bbox_inches="tight")

#### Looking into single predictions
Those are with a reasonably high confidence Score.

In [ ]:
thumbnails_folder = Path("thumbnails")
thumbnails = sorted(thumbnails_folder.glob(f"{image_stem}._*.JPG"))

print(f"Found {len(thumbnails)} thumbnails")

n_cols = 4
n_rows = (len(thumbnails) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten() if len(thumbnails) > 1 else [axes]

for ax, thumb_path in zip(axes, thumbnails):
    ax.imshow(Image.open(thumb_path))
    ax.set_title(thumb_path.name, fontsize=8)
    ax.axis("off")

for ax in axes[len(thumbnails):]:
    ax.axis("off")

plt.suptitle(f"{image_stem} ({len(thumbnails)} thumbnails)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{image_stem}_thumbnails.png", dpi=150, bbox_inches="tight")

### Manual Patch creation
Patch larger images into trainable tiles

In [ ]:
import pandas as pd
from PIL import Image as PILImage
from animaloc.data.patches import AnnotatedImageToPatches, AnnotationsFromCSV, save_batch_images

def create_patches(image_folder, annot_csv, patch_size, overlap, output_folder, min_visibility=0.0):
    """
    Patch a folder of annotated images into fixed-size tiles.

    Replaces the old animaloc Patcher API (removed in the git version) with
    the current AnnotatedImageToPatches per-image interface.
    """
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    annot_df = pd.read_csv(annot_csv)
    all_rows = []

    for img_name in annot_df["images"].unique():
        img_df = annot_df[annot_df["images"] == img_name]
        img_path = Path(image_folder) / img_name
        if not img_path.exists():
            print(f"  skipping missing image: {img_path}")
            continue

        img = PILImage.open(img_path).convert("RGB")
        annotations = AnnotationsFromCSV(img_df)
        patcher = AnnotatedImageToPatches(img, annotations, size=patch_size, overlap=overlap)
        patches, patch_annos = patcher.make_annotated_patches(min_visibility=min_visibility)

        save_batch_images(patches, img_name, str(output_path))
        all_rows.extend(list(patch_annos))

    gt_df = pd.DataFrame(all_rows)
    gt_csv = output_path / "gt.csv"
    gt_df.to_csv(gt_csv, index=False)
    print(f"Saved {len(gt_df)} annotations across {len(annot_df['images'].unique())} images → {gt_csv}")


if not Path("./data/val_patches").exists():
    create_patches(
        image_folder="./data/val",
        annot_csv="./data/val.csv",
        patch_size=(512, 512),
        overlap=0,
        output_folder="./data/val_patches",
        min_visibility=0.0,
    )
else:
    print("Validation patches already exist")

In [ ]:
train_dataloader = DataLoader(dataset=train_dataset, batch_size=12, shuffle=True, num_workers=num_workers)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=1, shuffle=False, num_workers=num_workers)

In [ ]:
lr = 1e-5
weight_decay = 1e-3
epochs = 2

optimizer = Adam(params=herdnet.parameters(), lr=lr, weight_decay=weight_decay)

In [ ]:
trainer = Trainer(
    model=herdnet,
    train_dataloader=train_dataloader,
    optimizer=optimizer,
    num_epochs=epochs,
    evaluator=evaluator,
    work_dir=work_dir
    )

### Training
(about 15 minutes per epoch on RTX 4090 )

In [ ]:
# Start the training, the Focal and Cross Entropy Loss should already be very low from the start.
trainer.start(warmup_iters=100, checkpoints='best', select='max', validate_on='f1_score')


### Evaluate the just trained model

In [ ]:
# Load best checkpoint
pth_path = './output/best_model.pth'

herdnet_trained = load_model(herdnet, pth_path=pth_path)
herdnet_trained = LossWrapper(herdnet, losses=losses)

In [ ]:
test_dataset = CSVDataset(
    csv_file='./data/test_sample.csv',
    root_dir='./data/test_sample',
    albu_transforms=[A.Normalize(p=1.0)],
    end_transforms=[DownSample(down_ratio=down_ratio, anno_type='point')],
)

test_dataloader = DataLoader(dataset=test_dataset, batch_size=1, shuffle=False, num_workers=num_workers)

In [ ]:
# Create an Evaluator
test_evaluator = HerdNetEvaluator(
    model=herdnet_trained,
    dataloader=test_dataloader,
    metrics=metrics,
    stitcher=stitcher,
    work_dir=test_dir,
    header='test'
    )

In [ ]:
# Print global F1 score (%)
test_f1_score = test_evaluator.evaluate(returns='f1_score')
print(f"F1 score = {test_f1_score * 100:0.0f}%")

### Reproducable Training based on a Config
This training and inference can be based on the same configuration. This reduces the setup.

In [ ]:
# Clear any previous Hydra state (important when re-running cells in a notebook)
GlobalHydra.instance().clear()

config_dir = str(Path("models/general_2022/").resolve())

with initialize_config_dir(config_dir=config_dir, version_base="1.1"):
    cfg = compose(config_name="config", overrides=[
        "training_settings.epochs=1",
        "training_settings.batch_size=24",
        "model.load_from=models/general_2022/20220413_HerdNet_General_dataset_2022.pth",
        "datasets.train.csv_file=data/train_patches.csv",
        "datasets.train.root_dir=data/train_patches",
        "datasets.validate.csv_file=data/val_patches/gt.csv",
        "datasets.validate.root_dir=data/val_patches",
        "wandb_flag=true",
    ])

In [ ]:
# Run training using 
result = main(cfg)

### Visualisation
#### Training Data
![Augmented Example](./visualizations/augmented_dataset_example_3.png)

#### Heatmap on Validation
![Heatmap](./visualizations/heatmap_overlay_005c952ba7a612c40986806cc84a87e1573ef4f2_14.JPG_1.png)

#### Heatmap and Ground Truth
![Heatmap](./visualizations/heatmap_target_005c952ba7a612c40986806cc84a87e1573ef4f2_14.JPG_1.png)